# Week 4 — Human Activity Recognition: Small Temporal Transformer

Trains the model defined in `backend/app/ml/har_model.py` on the dataset plan
described below, and writes a checkpoint to the path in `.env`
(`HAR_CHECKPOINT_PATH`, default `models/har_temporal_transformer.pt`) that
`backend/pipeline/har.py` loads for inference.

### 5/5 Target Class Coverage (HMDB51 + Custom Clips)
We use the **HMDB51** action dataset (Serre Lab / Brown University) via the automated downloader
at `backend/scripts/download_har_data.py`, providing 100% direct coverage across all 5 target classes:
- `walking` (548 clips from HMDB51 `walk`)
- `running` (232 clips from HMDB51 `run`)
- `standing` (154 clips from HMDB51 `stand`)
- `sitting` (142 clips from HMDB51 `sit`)
- `climbing` (112 clips from HMDB51 `climb_stairs`)

Custom project clips can also be added into any folder in `data/raw/<class>/`.

## 1. Dataset Plan & Setup

| Target Class | HMDB51 Source | Available Clips | Status |
|---|---|---|---|
| `walking` | `walk` | 548 | Full coverage |
| `running` | `run` | 232 | Full coverage |
| `standing` | `stand` | 154 | Full coverage |
| `sitting` | `sit` | 142 | Full coverage |
| `climbing` | `climb_stairs` | 112 | Full coverage |

### Folder Convention
```
data/raw/
  walking/    clip_001.avi  clip_002.avi  ...
  running/    clip_001.avi  ...
  standing/   clip_001.avi  ...
  sitting/    clip_001.avi  ...
  climbing/   clip_001.avi  ...
```
If `data/raw/` is not yet populated, run `python backend/scripts/download_har_data.py` to download and organize the dataset.

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
from collections import defaultdict

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

from backend.app.core.config import get_settings
from backend.pipeline.ingestion import VideoSource
from backend.pipeline.detector import PersonTracker
from backend.pipeline.pose import PoseEstimator
from backend.pipeline.features import TemporalFeatureManager, SEQUENCE_LENGTH
from backend.app.ml.har_model import build_model, ACTIVITY_CLASSES, NUM_CLASSES

settings = get_settings()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CHECKPOINT_PATH = PROJECT_ROOT / settings.HAR_CHECKPOINT_PATH

print("Project root:", PROJECT_ROOT)
print("Classes:", ACTIVITY_CLASSES)
print("Raw clips expected at:", RAW_DIR)
print("Cached features will be written to:", FEATURES_DIR)

Project root: c:\Users\athar\Downloads\Development of a Multimodal Vision-Language
Classes: ['walking', 'running', 'standing', 'sitting', 'climbing']
Raw clips expected at: c:\Users\athar\Downloads\Development of a Multimodal Vision-Language\data\raw
Cached features will be written to: c:\Users\athar\Downloads\Development of a Multimodal Vision-Language\data\features


## 2. Extract (32, 205) feature sequences from raw clips

Reuses the exact Week 1–3 pipeline — `VideoSource` → `PersonTracker` →
`PoseEstimator` → `TemporalFeatureManager` — so the features a clip
produces here are pixel-for-pixel the same kind of feature the live
system will compute at inference time. No separate "offline" feature
logic to drift out of sync with the real pipeline.

Each clip yields **one or more** (32, 205) windows: once the buffer for
the single dominant person in a clip fills to 32 frames, we snapshot it
as one training example and keep going (with a stride) rather than only
taking one window per clip.

In [8]:
def extract_windows_from_video(video_path: Path, stride: int = 8, max_windows_per_clip: int = 6):
    """Runs the full Week1-3 pipeline over one clip and returns a list of
    (32, 205) numpy arrays. Assumes one primary person per clip — if
    multiple people are tracked, the highest-confidence track_id is used."""
    source = VideoSource(str(video_path), source_fps=30, processing_fps=12)
    if not source.open():
        print(f"  [skip] could not open {video_path.name}")
        return []

    tracker = PersonTracker(
        model_path=settings.DETECTOR_MODEL,
        device=settings.DETECTOR_DEVICE,
        conf_threshold=settings.DETECTOR_CONF_THRESHOLD,
        img_size=settings.DETECTOR_IMG_SIZE,
        tracker_config=settings.TRACKER_CONFIG,
    )
    pose_estimator = PoseEstimator()
    feature_manager = TemporalFeatureManager()

    windows = []
    primary_track_id = None
    frame_i = 0

    while True:
        result = source.read()
        if not result.ok:
            break
        frame = result.frame
        h, w = frame.shape[:2]
        tracked = tracker.track_frame(frame)
        if not tracked:
            frame_i += 1
            continue

        if primary_track_id is None:
            primary_track_id = max(tracked, key=lambda p: p.confidence).track_id
        person = next((p for p in tracked if p.track_id == primary_track_id), None)
        if person is None:
            frame_i += 1
            continue

        x1, y1, x2, y2 = (max(0, int(v)) for v in person.bbox_xyxy)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 - x1 < 10 or y2 - y1 < 10:
            frame_i += 1
            continue

        crop = frame[y1:y2, x1:x2]
        pose = pose_estimator.estimate(crop)
        if pose is None:
            frame_i += 1
            continue

        buf = feature_manager.update(person.track_id, pose, person.bbox_xyxy, (w, h))
        if buf.is_full and frame_i % stride == 0:
            windows.append(buf.as_array().copy())
            if len(windows) >= max_windows_per_clip:
                break
        frame_i += 1

    source.release()
    pose_estimator.close()
    return windows


def build_feature_cache(max_clips_per_class: int = 70):
    """Walks data/raw/<class>/*.(avi|mp4), extracts windows, saves each as its
    own .npy under data/features/<class>/. Skips already-cached clips."""
    counts = defaultdict(int)
    for class_name in ACTIVITY_CLASSES:
        class_raw_dir = RAW_DIR / class_name
        class_feat_dir = FEATURES_DIR / class_name
        class_feat_dir.mkdir(parents=True, exist_ok=True)

        if not class_raw_dir.exists():
            print(f"[WARNING] no raw clips folder for '{class_name}' at {class_raw_dir} — skipping")
            continue

        clips = sorted(class_raw_dir.glob("*.avi")) + sorted(class_raw_dir.glob("*.mp4"))
        if max_clips_per_class > 0:
            clips = clips[:max_clips_per_class]
        print(f"'{class_name}': {len(clips)} clips selected for extraction")
        for clip_path in clips:
            existing = list(class_feat_dir.glob(f"{clip_path.stem}_*.npy"))
            if existing:
                counts[class_name] += len(existing)
                continue
            windows = extract_windows_from_video(clip_path)
            for i, w in enumerate(windows):
                out_path = class_feat_dir / f"{clip_path.stem}_{i:02d}.npy"
                np.save(out_path, w)
            counts[class_name] += len(windows)
            print(f"  {clip_path.name}: {len(windows)} windows")

    print("\n=== Feature extraction summary ===")
    for class_name in ACTIVITY_CLASSES:
        print(f"  {class_name:12s}: {counts[class_name]} windows cached")
    return counts

# Run extraction (or use already cached features):
# feature_counts = build_feature_cache()

## 3. Dataset class + group-aware train/val/test split

**Why group-aware, not a plain random split:** multiple windows are
extracted from the *same clip* (via sliding stride). A random
split could put windows from the same clip in both train and test,
causing data leakage. The split below groups by clip filename so an entire clip's windows land
in exactly one of train/val/test.

In [9]:
class ActivitySequenceDataset(Dataset):
    def __init__(self, file_paths: list[Path], labels: list[int]):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)
 
    def __getitem__(self, idx):
        seq = np.load(self.file_paths[idx]).astype(np.float32)
        return torch.from_numpy(seq), self.labels[idx]


def load_cached_dataset(val_frac: float = 0.15, test_frac: float = 0.15, seed: int = 42):
    rng = np.random.default_rng(seed)
    all_files, all_labels, all_groups = [], [], []

    for class_idx, class_name in enumerate(ACTIVITY_CLASSES):
        class_dir = FEATURES_DIR / class_name
        for f in sorted(class_dir.glob("*.npy")):
            all_files.append(f)
            all_labels.append(class_idx)
            all_groups.append(f.stem.rsplit("_", 1)[0])  # clip name without window suffix

    if not all_files:
        raise RuntimeError(f"No cached features found in {FEATURES_DIR}. Run build_feature_cache() first!")

    groups = sorted(set(all_groups))
    rng.shuffle(groups)
    n_val = max(1, int(len(groups) * val_frac))
    n_test = max(1, int(len(groups) * test_frac))
    val_groups = set(groups[:n_val])
    test_groups = set(groups[n_val:n_val + n_test])
    train_groups = set(groups[n_val + n_test:])

    def subset(target_groups):
        files = [f for f, g in zip(all_files, all_groups) if g in target_groups]
        labels = [l for l, g in zip(all_labels, all_groups) if g in target_groups]
        return files, labels

    train_files, train_labels = subset(train_groups)
    val_files, val_labels = subset(val_groups)
    test_files, test_labels = subset(test_groups)

    print(f"Train: {len(train_files)} windows from {len(train_groups)} clips")
    print(f"Val:   {len(val_files)} windows from {len(val_groups)} clips")
    print(f"Test:  {len(test_files)} windows from {len(test_groups)} clips")
    print("\nPer-class distribution (train):")
    for class_idx, class_name in enumerate(ACTIVITY_CLASSES):
        n = sum(1 for l in train_labels if l == class_idx)
        print(f"  {class_name:12s}: {n}")

    return (
        ActivitySequenceDataset(train_files, train_labels),
        ActivitySequenceDataset(val_files, val_labels),
        ActivitySequenceDataset(test_files, test_labels),
    )

# train_ds, val_ds, test_ds = load_cached_dataset()

## 4. Training Loop

AdamW + cross-entropy with **class-weighted loss** to account for any class frequency imbalance,
plus early stopping on validation loss to prevent overfitting.

In [10]:
def compute_class_weights(train_ds: ActivitySequenceDataset) -> torch.Tensor:
    counts = np.bincount(train_ds.labels, minlength=NUM_CLASSES).astype(np.float32)
    counts = np.clip(counts, 1, None)
    weights = counts.sum() / (NUM_CLASSES * counts)
    return torch.tensor(weights, dtype=torch.float32)


def train_model(train_ds, val_ds, epochs: int = 40, batch_size: int = 32, lr: float = 3e-4, patience: int = 8):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Training on: {device.upper()}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=(len(train_ds) > batch_size))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = build_model().to(device)
    class_weights = compute_class_weights(train_ds).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
        train_loss /= len(train_ds)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item() * x.size(0)
                correct += (logits.argmax(dim=1) == y).sum().item()
        val_loss /= len(val_ds)
        val_acc = correct / len(val_ds)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1:3d}/{epochs:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save({
                "model_state_dict": model.state_dict(),
                "activity_classes": ACTIVITY_CLASSES,
                "val_accuracy": val_acc,
                "val_loss": val_loss,
                "epoch": epoch + 1,
            }, CHECKPOINT_PATH)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch+1} (no val_loss improvement for {patience} epochs)")
                break

    return model, history

# model, history = train_model(train_ds, val_ds)

## 5. Evaluation (SRS §12.3: Accuracy, Macro F1, Per-Class Precision/Recall, Confusion Matrix)

In [11]:
def evaluate_model(checkpoint_path: Path, test_ds: ActivitySequenceDataset):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model = build_model().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(y.numpy().tolist())

    print("Checkpoint epoch:", checkpoint.get("epoch"))
    print("Checkpoint val_accuracy at save time:", checkpoint.get("val_accuracy"))
    print("\n=== Test set classification report ===")
    print(classification_report(all_labels, all_preds, target_names=ACTIVITY_CLASSES, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(ACTIVITY_CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(ACTIVITY_CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("HAR Confusion Matrix (Test Set)")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
    return cm

# evaluate_model(CHECKPOINT_PATH, test_ds)

## 6. Reproducibility Manifest (NFR-12)

Writes a metadata JSON alongside the checkpoint recording exact training parameters, classes, dataset provenance, and validation metrics.

In [12]:
def write_manifest(checkpoint_path: Path, feature_counts: dict, history: dict):
    manifest = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "model": {
            "architecture": "TemporalActivityTransformer",
            "d_model": 128, "nhead": 4, "num_layers": 2, "dim_feedforward": 256,
            "input_dim": 205, "sequence_length": 32,
        },
        "classes": ACTIVITY_CLASSES,
        "dataset": {
            "source": "HMDB51 dataset (5 mapped classes: walk, run, stand, sit, climb_stairs) + custom clips",
            "per_class_window_counts": feature_counts,
        },
        "training": {
            "final_val_accuracy": history["val_acc"][-1] if history.get("val_acc") else None,
            "epochs_run": len(history.get("train_loss", [])),
        },
        "feature_schema_version": "week3-205dim-v1",
    }
    manifest_path = checkpoint_path.with_suffix(".manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest written to {manifest_path}")

# write_manifest(CHECKPOINT_PATH, feature_counts, history)